# TensorFlow 2.3 · 网络模块

按代码单元格的原编号查阅课程手册。

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

linear = layers.Dense(3)
input1 = tf.random.uniform((10, 64))
output = linear(input1)
print(output.shape)  # (10, 3)
input2 = tf.random.uniform((10, 16, 64))
output2 = linear(input2)
print(output2.shape)  # (10, 16, 3)

In [ ]:
conv1 = layers.Conv1D(10, kernel_size=2, strides=1, padding='valid')
input1 = tf.random.normal((32, 32, 256))  # (批次, 长度, 通道)
out = conv1(input1)
print(out.shape)  # (32, 31, 10)

In [ ]:
x = tf.random.normal((3, 5, 4, 1))  # (批次, 高, 宽, 通道)
conv = layers.Conv2D(4, (2, 3))
output = conv(x)
print(output.shape)  # (3, 4, 2, 4)

In [ ]:
batch_norm = layers.BatchNormalization(momentum=0.9, epsilon=1e-5)
input1 = tf.random.normal((2, 2))
output = batch_norm(input1, training=True)
print(input1, output)

In [ ]:
batch_norm = layers.BatchNormalization(momentum=0.9, epsilon=1e-5)
input1 = tf.random.normal((1, 2, 2, 2))
output = batch_norm(input1, training=True)
print(input1, output)

In [ ]:
feature_size = 32
num_steps = 35
batch_size = 2
num_hiddens = 2
X = tf.random.uniform((batch_size, num_steps, feature_size))
rnn = layers.SimpleRNN(num_hiddens, return_sequences=True, return_state=True)
Y, state = rnn(X)
print(X.shape, Y.shape, state.shape)

In [ ]:
# 四层 LSTM：每批 3 句、每句 5 个词、每词 10 个特征。
out = tf.random.normal((3, 5, 10))
hidden_states, cell_states = [], []
for _ in range(4):
    lstm = layers.LSTM(20, return_sequences=True, return_state=True)
    out, hidden, cell = lstm(out)
    hidden_states.append(hidden)
    cell_states.append(cell)

h = tf.stack(hidden_states)
c = tf.stack(cell_states)
print(out.shape)  # (3, 5, 20)
print(h.shape, c.shape)  # (4, 3, 20), (4, 3, 20)

In [ ]:
# 步长 3 的转置卷积展开到 24 点，裁去起始 1 点后为 23 点。
dconv1 = tf.keras.Sequential([
    layers.Conv1DTranspose(1, kernel_size=3, strides=3, padding='valid'),
    layers.Cropping1D(cropping=(1, 0)),
])
x = tf.random.normal((16, 8, 1))
print(x.shape, dconv1(x).shape)  # (16, 8, 1) → (16, 23, 1)

In [ ]:
dconv2 = tf.keras.Sequential([
    layers.Conv2DTranspose(1, kernel_size=3, strides=3, padding='valid'),
    layers.Cropping2D(cropping=((1, 0), (1, 0))),
])
x = tf.random.normal((16, 8, 8, 1))
print(x.shape, dconv2(x).shape)  # (16, 8, 8, 1) → (16, 23, 23, 1)

In [ ]:
class A():
    def __call__(self):
        print('Python is useful')
a = A()
a()  # 输出结果为：Python is useful

In [ ]:
class Cat:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __repr__(self):
        return f"{self.__class__.__name__}({self.name}, {self.age})"

my_Cat = Cat("petty", 3)
print(my_Cat)  # 输出结果为：Cat(petty, 3)

In [ ]:
class Cat:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __repr__(self):
        return f"{self.__class__.__name__}({self.name}, {self.age})"

print(dir(Cat))
# 输出结果为：
#  ['__class__', '__delattr__', '__dict__', '__dir__', 
#  '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__',
#  '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', 
#  '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', 
#  '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__']

In [ ]:
class MyNet(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.linear1 = layers.Dense(1024, activation='relu')
        self.batchnorm1 = layers.BatchNormalization(momentum=0.9, epsilon=1e-5)
        self.linear2 = layers.Dense(7 * 7 * 128, activation='relu')
        self.batchnorm2 = layers.BatchNormalization(momentum=0.9, epsilon=1e-5)
        self.reshape_features = layers.Reshape((7, 7, 128))
        self.conv = layers.Conv2DTranspose(64, 4, strides=2, padding='same')

    def call(self, x, training=False):
        x = self.batchnorm1(self.linear1(x), training=training)
        x = self.batchnorm2(self.linear2(x), training=training)
        return self.conv(self.reshape_features(x))


model = MyNet()
print(model(tf.zeros((2, 96))).shape)  # (2, 14, 14, 64)

In [ ]:
class MyNet(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.fc = tf.keras.Sequential([
            layers.Dense(1024, activation='relu'),
            layers.BatchNormalization(momentum=0.9, epsilon=1e-5),
            layers.Dense(7 * 7 * 128, activation='relu'),
            layers.BatchNormalization(momentum=0.9, epsilon=1e-5),
            layers.Reshape((7, 7, 128)),
        ])
        self.conv = tf.keras.Sequential([
            layers.Conv2DTranspose(64, 4, strides=2, padding='same', activation='relu'),
            layers.BatchNormalization(momentum=0.9, epsilon=1e-5),
            layers.Conv2DTranspose(1, 4, strides=2, padding='same', activation='tanh'),
        ])

    def call(self, x, training=False):
        return self.conv(self.fc(x, training=training), training=training)

In [ ]:
model = MyNet()
input_tensor = tf.random.uniform((32, 96))
target = model(input_tensor)
print(target.shape)  # (32, 28, 28, 1)

In [ ]:
class MyNet(tf.keras.Model):
    def __init__(self, in_dim, n_hidden_1, n_hidden_2, out_dim):
        super().__init__()
        self.network = tf.keras.Sequential([
            layers.Input((in_dim,)),
            layers.Dense(n_hidden_1),
            layers.BatchNormalization(momentum=0.9, epsilon=1e-5),
            layers.ReLU(),
            layers.Dense(n_hidden_2),
            layers.BatchNormalization(momentum=0.9, epsilon=1e-5),
            layers.ReLU(),
            layers.Dense(out_dim),
        ])

    def call(self, x, training=False):
        return self.network(x, training=training)


model = MyNet(28 * 28, 300, 100, 10)
input_tensor = tf.random.uniform((10, 28 * 28))
target = model(input_tensor)
print(target.shape)  # (10, 10)